## Palm Detection and Hand Landmarking using Mediapipe [new Task APIs] and Open CV [with 3 Running modes: Image, Video, Live Stream]

#### Downloading the `hand_landmarker.task` file if it does not exist


In [1]:
model_path = "../mediapipe_tasks/"

face_landmarker_models = [
    "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task",
    "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task",
]

face_stylizer_models = [
    "https://storage.googleapis.com/mediapipe-models/face_stylizer/blaze_face_stylizer/float32/1/blaze_face_stylizer.task",
    "https://storage.googleapis.com/mediapipe-models/face_stylizer/blaze_face_stylizer/float32/1/face_stylizer_color_ink.task",
    "https://storage.googleapis.com/mediapipe-models/face_stylizer/blaze_face_stylizer/float32/1/face_stylizer_color_sketch.task",
    "https://storage.googleapis.com/mediapipe-models/face_stylizer/blaze_face_stylizer/float32/1/face_stylizer_oil_painting.task",
    "https://storage.googleapis.com/mediapipe-models/face_stylizer/blaze_face_stylizer/float32/latest/blaze_face_stylizer.task",
    "https://storage.googleapis.com/mediapipe-models/face_stylizer/blaze_face_stylizer/float32/latest/face_stylizer_color_ink.task",
    "https://storage.googleapis.com/mediapipe-models/face_stylizer/blaze_face_stylizer/float32/latest/face_stylizer_color_sketch.task",
    "https://storage.googleapis.com/mediapipe-models/face_stylizer/blaze_face_stylizer/float32/latest/face_stylizer_oil_painting.task",
]

gesture_recognizer_models = [
    "https://storage.googleapis.com/mediapipe-models/gesture_recognizer/gesture_recognizer/float16/1/gesture_recognizer.task",
    "https://storage.googleapis.com/mediapipe-models/gesture_recognizer/gesture_recognizer/float16/latest/gesture_recognizer.task",
]

hand_landmarker_models = [
    "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task",
    "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task",
]

holistic_landmarker_models = [
    "https://storage.googleapis.com/mediapipe-models/holistic_landmarker/holistic_landmarker/float16/1/holistic_landmarker.task",
    "https://storage.googleapis.com/mediapipe-models/holistic_landmarker/holistic_landmarker/float16/latest/holistic_landmarker.task",
]

image_generator_models = [
    "https://storage.googleapis.com/mediapipe-models/image_generator/LoRA_weights/pokemon_lora.task",
    "https://storage.googleapis.com/mediapipe-models/image_generator/LoRA_weights/teapot_lora.task",
]

pose_landmarker_models = [
    "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/1/pose_landmarker_full.task",
    "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/latest/pose_landmarker_full.task",
    "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_heavy/float16/1/pose_landmarker_heavy.task",
    "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_heavy/float16/latest/pose_landmarker_heavy.task",
    "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task",
    "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/latest/pose_landmarker_lite.task",
]

In [2]:
import os
import wget

model_uri = hand_landmarker_models[1]
model_name = model_uri.split("/")[-1]
model_path = os.path.join("../mediapipe/tasks/", model_name)

if os.path.exists(model_path):
    print(f"{model_name} already exists at: {model_path}")
else:
    taskfile = wget.download(model_uri, out=model_path)
    print(f"Downloaded: {taskfile}")

Downloaded: ../mediapipe/tasks/hand_landmarker.task


#### Defining Helper Functions


1. To check if opencv is detecting the webcam feed


In [3]:
import cv2


def start_camera_stream(device_index=0):
    """
    Opens the camera feed and displays it.
    Press 'q' to close the window.
    """
    cap = cv2.VideoCapture(device_index)

    if not cap.isOpened():
        print(f"Error: Could not open camera at index {device_index}")
        return

    print("Camera started. Press 'q' to exit.")

    while True:
        ret, frame = cap.read()
        if not ret:
            print("Error: Failed to grab frame.")
            break

        cv2.imshow("Camera Feed", frame)

        # Wait for 1ms and check for 'q' key
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    # Clean up resources
    cap.release()
    cv2.destroyAllWindows()
    print("Camera stream closed.")

In [4]:
# start_camera_stream()

2. Function to draw hand landmarks


In [5]:
import numpy as np
import mediapipe as mp

mp_hands = mp.tasks.vision.HandLandmarksConnections
mp_drawing = mp.tasks.vision.drawing_utils
mp_drawing_styles = mp.tasks.vision.drawing_styles

MARGIN = 10  # pixels
FONT_SIZE = 1
FONT_THICKNESS = 1
HANDEDNESS_TEXT_COLOR = (88, 205, 54)  # vibrant green


def draw_landmarks_on_image(rgb_image, detection_result):
    hand_landmarks_list = detection_result.hand_landmarks
    handedness_list = detection_result.handedness
    annotated_image = np.copy(rgb_image)

    # Loop through the detected hands to visualize.
    for idx in range(len(hand_landmarks_list)):
        hand_landmark = hand_landmarks_list[idx]
        handedness = handedness_list[idx]

        # Draw the hand landmarks.
        mp_drawing.draw_landmarks(
            annotated_image,
            hand_landmark,
            mp_hands.HAND_CONNECTIONS,
            mp_drawing_styles.get_default_hand_landmarks_style(),
            mp_drawing_styles.get_default_hand_connections_style(),
        )

        # Get the top left corner of the detected hand's bounding box.
        height, width, _ = annotated_image.shape
        x_coordinates = [landmark.x for landmark in hand_landmarks]
        y_coordinates = [landmark.y for landmark in hand_landmarks]
        text_x = int(min(x_coordinates) * width)
        text_y = int(min(y_coordinates) * height) - MARGIN

        # Draw handedness (left or right hand) on the image.
        cv2.putText(
            annotated_image,
            f"{handedness[0].category_name}",
            (text_x, text_y),
            cv2.FONT_HERSHEY_DUPLEX,
            FONT_SIZE,
            HANDEDNESS_TEXT_COLOR,
            FONT_THICKNESS,
            cv2.LINE_AA,
        )

    return annotated_image

3. Function to count fingers


In [6]:
def count_fingers(hand_landmarks):
    fingers = []
    # fingertip indices for thumb, index, middle, ring, and pinky fingers (fi)
    fingertip_indices = [8, 12, 16, 20]

    # lower joints for thumb, index, middle, ring, and pinky fingers (ljin = fin -2)
    lower_joint_indices = [6, 10, 14, 18]
    fingers.append(hand_landmarks[4].x < hand_landmarks[3].x)
    for tip_idx, joint_idx in zip(fingertip_indices, lower_joint_indices):
        fingers.append(hand_landmarks[tip_idx].y < hand_landmarks[joint_idx].y)

    return fingers.count(True)

In [7]:
def result_callback(result, output_image, timestamp_ms):
    annotated_image = draw_landmarks_on_image(output_image, result)
    cv2.imshow("Hand Tracking", cv2.cvtColor(
        annotated_image, cv2.COLOR_RGB2BGR))

#### Configuring the HandLandmarker options to process live stream


In [8]:
import mediapipe as mp

from mediapipe.tasks.python.core.base_options import BaseOptions
from mediapipe.tasks.python.vision import RunningMode
from mediapipe.tasks.python.vision import (
    HandLandmarker,
    HandLandmarkerOptions,
    HandLandmarkerResult,
)

In [9]:

base_options = BaseOptions(model_asset_path=model_path)
options = HandLandmarkerOptions(
    base_options=base_options,
    running_mode=RunningMode.IMAGE,
    num_hands=2,
    min_hand_detection_confidence=0.7,
    min_hand_presence_confidence=0.7,
    min_tracking_confidence=0.5,
    # result_callback=result_callback,
)
hand_landmarker = HandLandmarker.create_from_options(options)

I0000 00:00:1778483625.832870  207486 init-domain.cc:128] Fiber init: default domain = pthread, concurrency = 8, prefix = pthread-default
I0000 00:00:1778483626.012673  207486 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M1
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1778483626.025931  207498 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778483626.048795  207498 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


#### Starting our Detection Loop


In [10]:
mp_hands = mp.tasks.vision.HandLandmarksConnections
mp_drawing = mp.tasks.vision.drawing_utils
mp_drawing_styles = mp.tasks.vision.drawing_styles

cap = cv2.VideoCapture(0)
while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
    # hand_landmarker.detect_async(mp_image, timestamp_ms=0)
    result = hand_landmarker.detect(mp_image)

    if result.hand_landmarks:
        annotated_image = np.copy(mp_image.numpy_view())
        for i, (hand_landmark, hand_world_landmark) in enumerate(zip(result.hand_landmarks, result.hand_world_landmarks)):

            # Use 'i' to get the correct handedness for the current hand in the loop
            model_handedness = result.handedness[i][0].category_name

            # INVERT the handedness to account for the cv2.flip mirror effect
            corrected_handedness = "Right" if model_handedness == "Left" else "Left"

            finger_count = count_fingers(hand_landmark)
            # print(f"Number of fingers raised: {finger_count}")

            # Draw the hand landmarks.
            mp_drawing.draw_landmarks(
                frame,
                hand_landmark,
                mp_hands.HAND_CONNECTIONS,
                mp_drawing_styles.get_default_hand_landmarks_style(),
                mp_drawing_styles.get_default_hand_connections_style(),
            )

            height, width, color_channel = annotated_image.shape
            for lm in hand_landmark:
                cx, cy = int(lm.x * width), int(lm.y * height)
                cv2.circle(frame, (cx, cy), 5, (0, 255, 0), cv2.FILLED)

            y_offset = i * 20 

            cv2.putText(
                frame,
                f"Hand {i+1} ({corrected_handedness}): {finger_count} fingers", 
                (20, 60 + y_offset),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 0, 255),
                2,
                cv2.LINE_AA,
            )

            cv2.putText(
                frame,
                f"Confidence Score: {result.handedness[0][0].score}",
                (10, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0, 255, 255),
                2,
                cv2.LINE_AA,
            )

            # cv2.putText(
            #     frame,
            #     f"Handedness: {corrected_handedness}",
            #     (10, 90),
            #     cv2.FONT_HERSHEY_SIMPLEX,
            #     1,
            #     (88, 205, 54),
            #     2,
            #     cv2.LINE_AA,
            # )
            # print(result)

    # Display the original frame (landmarks will be drawn in the callback).
    cv2.imshow("Hand Tracking", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

# Clean up resources
cap.release()
cv2.destroyAllWindows()
print("Camera stream closed.")

W0000 00:00:1778483628.361715  207499 landmark_projection_calculator.cc:81] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


Camera stream closed.
